# Classifications de CT-scans pulmonaires : modèle CNN ResNet50
## Chargement du dataset, préparation des données, entraînement du modèle puis classification en 4 classes : adenocarcinoma, large cell carcinoma, normal, squamous cell carcinoma

## Imports

In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import kagglehub

from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

## Chargement du dataset (Kaggle)

In [ ]:
dataset_path = kagglehub.dataset_download("mohamedhanyyy/chest-ctscan-images")
dataset_path = Path(dataset_path)

if (dataset_path / "Data").exists():
    data_root = dataset_path / "Data"
elif (dataset_path / "data").exists():
    data_root = dataset_path / "data"
else:
    raise FileNotFoundError("Impossible de trouver le dossier Data/data dans le dataset téléchargé.")

train_dir = data_root / "train"
val_dir = data_root / "valid"
test_dir = data_root / "test"

print("Dataset :", data_root)
print("Train existe :", train_dir.exists())
print("Valid existe :", val_dir.exists())
print("Test existe :", test_dir.exists())

## Uniformisation des noms de classes

In [ ]:
rename_map = {
    "adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib": "adenocarcinoma",
    "large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa": "large.cell.carcinoma",
    "squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa": "squamous.cell.carcinoma",
}

for split_dir in [train_dir, val_dir]:
    for old_name, new_name in rename_map.items():
        old_path = split_dir / old_name
        new_path = split_dir / new_name

        if old_path.exists() and not new_path.exists():
            old_path.rename(new_path)

for split_name, split_dir in [("train", train_dir), ("valid", val_dir), ("test", test_dir)]:
    classes = sorted([p.name for p in split_dir.iterdir() if p.is_dir()])
    print(split_name, ":", classes)

## Pré-traitement des images (conversion de RGBA à RGB puis redimension en 224x224)

In [ ]:
train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Création des datasets et dataloaders

In [5]:
train_dataset = ImageFolder(train_dir, transform=train_transform)
val_dataset = ImageFolder(val_dir, transform=val_test_transform)
test_dataset = ImageFolder(test_dir, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes détectées :", train_dataset.classes)
print("Classe -> label :", train_dataset.class_to_idx)

print("Train :", len(train_dataset))
print("Valid :", len(val_dataset))
print("Test  :", len(test_dataset))

Classes détectées : ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']
Classe -> label : {'adenocarcinoma': 0, 'large.cell.carcinoma': 1, 'normal': 2, 'squamous.cell.carcinoma': 3}
Train : 613
Valid : 72
Test  : 315


## Vérification d'un batch

In [6]:
images, labels = next(iter(train_loader))

print("Images :", images.shape)
print("Labels :", labels.shape)
print("Exemple labels :", labels[:10])

Images : torch.Size([32, 3, 224, 224])
Labels : torch.Size([32])
Exemple labels : tensor([1, 1, 3, 0, 0, 1, 2, 0, 1, 0])


## Device 

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

Device utilisé : cpu


## Création du modèle ResNet50 (modèle pré-entraîné et modification de la dernière couche pour une classification en 4 classes)

In [8]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Entraînement du modèle 

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.4f}")
    print(f"Valid Loss : {val_loss:.4f} | Valid Acc : {val_acc:.4f}")
    print("-" * 50)

Epoch 1/5
Train Loss : 0.8681 | Train Acc : 0.6264
Valid Loss : 1.5594 | Valid Acc : 0.5000
--------------------------------------------------


## Test final

In [ ]:
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_loss = test_loss / test_total
test_acc = test_correct / test_total

print("Résultats sur test")
print(f"Test Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}")